# Chapter 14 &mdash; An RE Language: $L_{G1neG2}$, and the Shift from Syntax to Behaviour

**Concept 8 of the Chapter 14 decomposition:** *An RE Language: $L_{G1neG2}$, and the Shift from Syntax to Behaviour*

CFG inequivalence is semi-decidable by enumerating strings in numeric order &mdash; but equivalence is not.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-L-G1neG2-Is-RE/Concept-L-G1neG2-Is-RE.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$L_{G1neG2} = \{\langle G_1,G_2\rangle : L(G_1)\neq L(G_2)\}$$

This is **RE**, and the semi-decider is a search: enumerate strings in numeric order
(Chapter 5, `nthnumeric`), test membership in both grammars, and halt the moment they
disagree. If the languages differ, some string witnesses it and you will reach it.

If they are **equal** you search forever. And that is not a failure of imagination:
CFG **equivalence** is undecidable, so its complement here is genuinely not
semi-decidable by any method.

Notice the shift. $L_{EmptyDFA}$ (Concept 5) was a question about a machine's
**structure**. This is a question about a grammar's **behaviour** &mdash; the set of
strings it generates &mdash; and behaviour is unbounded.

## 2. Definitions

### The semi-decider

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps


def semi_decide_ne(G1, G2, budget=200):
    # enumerate strings in numeric order; halt at the first disagreement
    L1 = set(language(G1, 10))
    L2 = set(language(G2, 10))
    for i in range(budget):
        s = nthnumeric(i, ['a', 'b'])
        if len(s) > 10: break
        if (s in L1) != (s in L2):
            return ('DIFFERENT', i, s)
    return ('no answer yet', budget, None)

### Some grammar pairs

In [ ]:
G1 = mkg({'S': ["", "aSb"]})                     # a^n b^n
G2 = mkg({'S': ["", "aSb"]})                     # the same, written the same way
G3 = mkg({'S': ["", "aXb"], 'X': ["", "aXb"]})   # the same language, different grammar
G4 = mkg({'S': ["", "ab", "aSb"]})               # also the same language!
G5 = mkg({'S': ["", "aSbb"]})                    # a^n b^2n -- genuinely different

## 3. Tests

When the languages **differ**, the search halts with a witness.

In [ ]:
verdict, i, s = semi_decide_ne(G1, G5)
print("G1 vs G5 :", verdict, " at index", i, " witness", repr(s))
assert verdict == 'DIFFERENT'
print("  in L(G1)?", s in set(language(G1, 10)))
print("  in L(G5)?", s in set(language(G5, 10)))

When they are **equal**, it searches forever &mdash; here, until the budget runs out.

In [ ]:
for name, G in [('G2 (identical)', G2), ('G3 (same language)', G3),
                ('G4 (same again)', G4)]:
    verdict, i, s = semi_decide_ne(G1, G)
    print("  G1 vs %-20s -> %-16s after %d candidates" % (name, verdict, i))
    assert verdict == 'no answer yet'

Raising the budget does not help &mdash; it only postpones the silence.

In [ ]:
for b in [20, 100, 400]:
    verdict, i, _ = semi_decide_ne(G1, G3, budget=b)
    print("  budget %3d : %s" % (b, verdict))
print("\nEquality can never be confirmed by a search for a counterexample.")

Numeric order is the right enumeration: the shortest witness comes first.

In [ ]:
print("numeric order :", [nthnumeric(i, ['a','b']) for i in range(12)])
verdict, i, s = semi_decide_ne(G1, G5)
print("\nwitness found at index %d, length %d" % (i, len(s)))
print("A longer-first enumeration would have taken far longer to find it.")

**Syntax versus behaviour**, which is the lesson.

In [ ]:
print("L_EmptyDFA  : a question about a finite GRAPH        -> decidable")
print("L_G1neG2    : a question about generated BEHAVIOUR   -> RE only")
print()
print("CFG equivalence is undecidable, so the complement of L_G1neG2 is")
print("not RE either.  By Concept 7, that is why L_G1neG2 is not recursive.")

## 4. Exercises


1. Why is DFA equivalence decidable while CFG equivalence is not?
2. Adapt the semi-decider to $L_{G\neq\emptyset}$. Is that one decidable?
3. What goes wrong if you enumerate by length but in a random order within a length?

In [ ]:
# Your work for the exercises above.